# Boltz-2 screen for human GHS-R\n\nThis notebook runs the end-to-end GHS-R ligand screening workflow.

In [ ]:
!pip install -U boltz pandas matplotlib pyyaml py3Dmol

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Keep Boltz cache on the Colab VM disk, not Google Drive. Drive can raise
# OSError: [Errno 5] Input/output error on many small files (e.g. mols/*.pkl).
os.environ['BOLTZ_CACHE'] = '/content/boltz_cache'
os.makedirs(os.environ['BOLTZ_CACHE'], exist_ok=True)
print('BOLTZ_CACHE =', os.environ['BOLTZ_CACHE'])

In [ ]:
# Clone https://github.com/2000calories/ghsr_af3_molecule_screening (skip if already present).
# Alternative: upload your own zip and unzip into /content/ghsr_af3_molecule_screening.
import os
import subprocess

_repo_dir = '/content/ghsr_af3_molecule_screening'
_clone_url = 'https://github.com/2000calories/ghsr_af3_molecule_screening.git'
if os.path.isdir(os.path.join(_repo_dir, '.git')):
    print('Repo already present:', _repo_dir)
else:
    subprocess.run(['git', 'clone', _clone_url, _repo_dir], check=True)


In [ ]:
import os; os.chdir('/content/ghsr_af3_molecule_screening')

In [ ]:
!python scripts/build_inputs.py --library data/ligand_library.csv --target-fasta data/target_ghsr.fasta --out-dir inputs

In [ ]:
# run_screen defaults to --accelerator auto (GPU if CUDA, else CPU — avoids errors on CPU-only Colab).
# Resume-from-Drive: any ligand with predictions on Drive is copied into local outputs/ so
# run_screen.py's built-in is_completed() check skips it. After the run, only ligands with new
# predictions are pushed back per-folder (never wipes the Drive outputs/ tree).
import os
import shutil
import subprocess
from pathlib import Path

FORCE = False  # True -> skip Drive hydration and re-run every ligand (passes --force to run_screen.py)

DRIVE_BOLTZ = "/content/drive/MyDrive/ghsr_af3_molecule_screening/boltz_colab_results"
drive_outputs = Path(DRIVE_BOLTZ) / "outputs"
local_outputs = Path("outputs")
local_outputs.mkdir(parents=True, exist_ok=True)


def _has_predictions(ligand_dir: Path) -> bool:
    pred_dir = ligand_dir / "predictions"
    return pred_dir.is_dir() and any(pred_dir.rglob("*.json"))


hydrated = []
if not FORCE and drive_outputs.is_dir():
    for yaml_path in sorted(Path("inputs").glob("*.yaml")):
        ligand_id = yaml_path.stem
        src = drive_outputs / ligand_id
        if _has_predictions(src):
            shutil.copytree(src, local_outputs / ligand_id, dirs_exist_ok=True)
            hydrated.append(ligand_id)
print(f"Hydrated {len(hydrated)} ligand(s) from Drive; will skip them.")

cmd = [
    "python",
    "scripts/run_screen.py",
    "--inputs-dir",
    "inputs",
    "--outputs-dir",
    "outputs",
    "--recycling-steps",
    "3",
    "--diffusion-samples",
    "1",
    "--msa-mode",
    "server",
]
if FORCE:
    cmd.append("--force")
subprocess.run(cmd, check=True)

os.makedirs(drive_outputs, exist_ok=True)
synced = 0
for ligand_dir in sorted(p for p in local_outputs.iterdir() if p.is_dir()):
    if not _has_predictions(ligand_dir):
        continue
    shutil.copytree(ligand_dir, drive_outputs / ligand_dir.name, dirs_exist_ok=True)
    synced += 1
print(f"Synced {synced} ligand folder(s) to: {drive_outputs}")

In [ ]:
# Write ranking.csv, plots, top_poses, and outputs.zip to Google Drive (not the Colab VM disk).
# Default matches a Drive copy of this repo; change DRIVE_RESULTS if your project path differs.
import os
import subprocess

DRIVE_RESULTS = "/content/drive/MyDrive/ghsr_af3_molecule_screening/boltz_colab_results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
ranking_csv = f"{DRIVE_RESULTS}/ranking.csv"
plots_dir = f"{DRIVE_RESULTS}/plots"
top_poses_dir = f"{DRIVE_RESULTS}/top_poses"
zip_path = f"{DRIVE_RESULTS}/outputs.zip"

subprocess.run(
    [
        "python",
        "scripts/parse_results.py",
        "--library",
        "data/ligand_library.csv",
        "--outputs-dir",
        "outputs",
        "--results-dir",
        DRIVE_RESULTS,
    ],
    check=True,
)
subprocess.run(
    [
        "python",
        "scripts/visualize.py",
        "--ranking-csv",
        ranking_csv,
        "--outputs-dir",
        "outputs",
        "--plots-dir",
        plots_dir,
        "--top-poses-dir",
        top_poses_dir,
    ],
    check=True,
)
subprocess.run(["zip", "-r", zip_path, "outputs"], check=True)
os.environ["BOLTZ_RANKING_CSV"] = ranking_csv
os.environ["BOLTZ_TOP_POSES_DIR"] = top_poses_dir
print("Saved under:", DRIVE_RESULTS)

In [ ]:
import os
import pandas as pd

_ranking = "/content/drive/MyDrive/ghsr_af3_molecule_screening/boltz_colab_results/ranking.csv"
df = pd.read_csv(os.environ.get("BOLTZ_RANKING_CSV", _ranking))
df.head(20)

In [ ]:
import os
import py3Dmol
from pathlib import Path

_top_poses = "/content/drive/MyDrive/ghsr_af3_molecule_screening/boltz_colab_results/top_poses"
top_pose_dir = Path(os.environ.get("BOLTZ_TOP_POSES_DIR", _top_poses))
poses = sorted(top_pose_dir.glob('*.pdb'))
print('Found', len(poses), 'pose files')
if poses:
  model = poses[0].read_text()
  view = py3Dmol.view(width=900, height=600)
  view.addModel(model, 'pdb')
  view.setStyle({'cartoon': {'color': 'spectrum'}})
  view.setStyle({'resn': ['LIG']}, {'stick': {}})
  view.zoomTo()
  view.show()